# 01 — Prepare Held-Out Test Set

This notebook creates a held-out test set that is **never used during training**.
It is saved in two formats for the evaluation notebook:

- `test_set.csv` — human-readable with original string IDs and labels
- `test_set.npz` — precomputed numpy arrays for standalone evaluation

**Run this once** before training. The test set is fixed and reusable across experiments.

In [ ]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

sys.path.insert(0, str(Path.cwd()))
from pbi import quick_connect
from pbi.negative_examples import NegativeExampleGenerator
from pbi_adapter import PBIAdapter

## 1. Connect to PBI-Scope

In [ ]:
retriever = quick_connect()
print("Connected to PBI-Scope database")

## 2. Load and Classify All Pairs

In [ ]:
adapter = PBIAdapter(
    retriever,
    bacterium_threshold=7_000_000,
    phage_threshold=200_000,
    bacterium_min_length=150_000,
    phage_min_length=1_500,
)

all_pairs = adapter.get_pair_ids_only(shuffle=True)
print(f"Total pairs in database: {len(all_pairs)}")

positive_pairs, private_negatives = adapter.classify_pairs_by_interaction(all_pairs)
print(f"Positive pairs: {len(positive_pairs)}")
print(f"True negatives (private data): {len(private_negatives)}")

## 3. Generate Synthetic Negatives

In [ ]:
neg_gen = NegativeExampleGenerator(retriever)
generated_negatives = neg_gen.generate_random_negatives(
    positive_pairs, ratio=1.0
)

# Deduplicate against private negatives
if len(private_negatives) > 0:
    private_neg_set = set(
        zip(private_negatives["Phage_ID"], private_negatives["Host_ID"])
    )
    before = len(generated_negatives)
    generated_negatives = generated_negatives[
        ~generated_negatives.apply(
            lambda r: (r["Phage_ID"], r["Host_ID"]) in private_neg_set, axis=1
        )
    ].reset_index(drop=True)
    deduped = before - len(generated_negatives)
    if deduped > 0:
        print(f"Removed {deduped} duplicates against private negatives")

# Cap at 2x private negatives
max_generated = max(len(private_negatives) * 2, 100) if len(private_negatives) > 0 else len(positive_pairs)
if len(generated_negatives) > max_generated:
    generated_negatives = generated_negatives.sample(n=max_generated, random_state=42).reset_index(drop=True)
    print(f"Capped generated negatives to {max_generated}")

generated_negatives["negative_source"] = "generated"
print(f"Generated negatives: {len(generated_negatives)}")

## 4. Prepare Training Data

In [ ]:
# Combine negatives
if len(private_negatives) > 0 and len(generated_negatives) > 0:
    negative_pairs = pd.concat([private_negatives, generated_negatives], ignore_index=True)
elif len(private_negatives) > 0:
    negative_pairs = private_negatives
else:
    negative_pairs = generated_negatives

couples, labels, sources = adapter.prepare_training_data(positive_pairs, negative_pairs)
print(f"Total pairs: {len(couples)}")
print(f"  Positive: {int(labels.sum())}")
print(f"  Negative: {int(len(labels) - labels.sum())}")
print(f"Sources: {dict(zip(*np.unique(sources, return_counts=True)))}")

## 5. Split Off Held-Out Test Set (15%)

In [ ]:
stratify_key = np.array([
    "pos" if l == 1 else f"neg_{s}"
    for l, s in zip(labels, sources)
])

_, test_couples, _, test_labels, _, test_sources = train_test_split(
    couples, labels, sources,
    stratify=stratify_key, test_size=0.15, shuffle=True, random_state=42
)

print(f"Test set: {len(test_couples)} pairs")
print(f"  Sources: {dict(zip(*np.unique(test_sources, return_counts=True)))}")

## 6. Save Test Set

In [ ]:
# Build reverse ID maps to recover original string IDs
host_id_map, phage_id_map = adapter.get_id_maps()
reverse_host = {v: k for k, v in host_id_map.items()}
reverse_phage = {v: k for k, v in phage_id_map.items()}

test_df = pd.DataFrame({
    "Phage_ID": [reverse_phage[pid] for pid in test_couples[:, 1]],
    "Host_ID": [reverse_host[hid] for hid in test_couples[:, 0]],
    "label": test_labels.astype(int),
    "source": test_sources,
})

out_dir = Path.cwd() / "test_data"
out_dir.mkdir(exist_ok=True)

# Save CSV (human-readable)
test_df.to_csv(out_dir / "test_set.csv", index=False)
print(f"Saved test_set.csv ({len(test_df)} rows)")

# Save excluded pair IDs (for train.py --exclude-ids to prevent data leakage)
excluded_df = test_df[["Phage_ID", "Host_ID"]].copy()
excluded_df.to_csv(out_dir / "excluded_pairs.csv", index=False)
print(f"Saved excluded_pairs.csv ({len(excluded_df)} rows)")

# Save NPZ (precomputed arrays for evaluation notebook)
np.savez(
    out_dir / "test_set.npz",
    couples=test_couples,
    labels=test_labels,
    sources=test_sources,
)
print(f"Saved test_set.npz")

## Done

The held-out test set is ready. Files saved in `test_data/`:

| File | Contents |
|------|----------|
| `test_set.csv` | Phage_ID, Host_ID, label, source |
| `excluded_pairs.csv` | Phage_ID, Host_ID of test pairs (for `--exclude-ids` in train.py) |
| `test_set.npz` | `couples` (N,2), `labels` (N,), `sources` (N,) |

**Next steps:**
1. Train a model (excluding test pairs):
   ```bash
   python train.py --config config.yaml --exclude-ids test_data/excluded_pairs.csv
   ```
2. Evaluate: open `02_evaluate_model.ipynb`